In [2]:
import numpy as np

import pandas as pd

# Ganti path sesuai lokasi file Anda
df = pd.read_csv('data/Coinglass/data_ohlcv_price_-_binance_postprocessed w regime ONLY.csv')
df.head()


,date,open,high,low,close,volume_usd,open_basis,close_basis,open_change,close_change,...,taker_buy_volume_usd,taker_sell_volume_usd,top_account_long_percent,top_account_short_percent,top_account_long_short_ratio,top_position_long_percent,top_position_short_percent,top_position_long_short_ratio,whale_index_value,regime_wf
0,2024-04-24 12:00:00+00:00,66550.6,66711.9,66302.6,66317.8,4.479229e+08,0.0314,0.0531,20.92,35.22,...,2.122550e+08,2.356679e+08,59.15,40.85,1.45,54.00,46.00,1.17,-28.270,2
1,2024-04-24 13:00:00+00:00,66317.8,66325.2,65810.0,66040.9,1.430633e+09,0.0531,0.0213,35.21,14.09,...,6.680783e+08,7.625547e+08,59.14,40.86,1.45,53.99,46.01,1.17,-28.490,2
2,2024-04-24 14:00:00+00:00,66041.0,66180.0,64706.0,65212.7,3.163982e+09,0.0212,0.0340,14.00,22.15,...,1.456739e+09,1.707243e+09,59.81,40.19,1.49,54.10,45.90,1.18,-32.395,2
3,2024-04-24 15:00:00+00:00,65212.8,65221.1,64489.7,64594.0,1.694626e+09,0.0338,0.0249,22.06,16.11,...,8.367921e+08,8.578336e+08,62.24,37.76,1.65,54.90,45.10,1.22,-44.660,2
4,2024-04-24 16:00:00+00:00,64593.5,64947.6,64315.9,64771.4,1.430894e+09,0.0257,0.0411,16.60,26.60,...,7.222813e+08,7.086125e+08,63.27,36.73,1.72,55.36,44.64,1.24,-49.940,2


In [3]:
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))

df['ema_20'] = df['close'].ewm(span=20).mean()
df['ema_50'] = df['close'].ewm(span=50).mean()
df['ema_200'] = df['close'].ewm(span=200).mean()

df['ema_ratio_20_50'] = df['ema_20'] / df['ema_50']
df['ema_ratio_50_200'] = df['ema_50'] / df['ema_200']

df['range'] = (df['high'] - df['low']) / df['close']
df['range_rank_20'] = df['range'].rolling(20).rank(pct=True)

df['body'] = (df['close'] - df['open']) / df['close']
df['wick_up'] = (df['high'] - df[['open','close']].max(axis=1)) / df['close']
df['wick_dn'] = (df[['open','close']].min(axis=1) - df['low']) / df['close']

In [4]:
TP_PCT = 0.02
MAX_HOLD = 100

labels = []

for i in range(len(df)):

    if i + MAX_HOLD >= len(df):
        labels.append(np.nan)
        continue

    entry = df['close'].iloc[i]

    tp_long  = entry * (1 + TP_PCT)
    tp_short = entry * (1 - TP_PCT)

    label = 0  # default neutral

    for j in range(1, MAX_HOLD):

        future = df['close'].iloc[i + j]

        if future >= tp_long:
            label = 1   # long win
            break

        if future <= tp_short:
            label = -1  # short win
            break

    labels.append(label)

df['label'] = labels

In [5]:
features = [
    'log_ret',
    'ema_ratio_20_50',
    'ema_ratio_50_200',
    'range_rank_20',
    'body', 'wick_up', 'wick_dn'
]

df_ml = df.dropna().reset_index(drop=True)

X = df_ml[features]
y = df_ml['label']

print(y.value_counts(normalize=True))


label
 1.0    0.471110
-1.0    0.471047
 0.0    0.057842
Name: proportion, dtype: float64


In [6]:
import numpy as np
import pandas as pd
import xgboost as xgb

# =========================
# FEATURE SET
# =========================
features = [
    'log_ret',
    'ema_ratio_20_50',
    'ema_ratio_50_200',
    'range_rank_20',
    'body', 'wick_up', 'wick_dn'
]

df_ml = df.dropna().reset_index(drop=True)

X = df_ml[features]

# =========================
# LABEL ENCODING
# -1 → 0  (short)
#  0 → 1  (neutral)
#  1 → 2  (long)
# =========================
label_map = {-1: 0, 0: 1, 1: 2}
y = df_ml['label'].map(label_map)

print("Label distribution:")
print(y.value_counts(normalize=True))


# =========================
# WALK-FORWARD PARAMETERS
# =========================
TRAIN_SIZE = 24 * 30 * 12
TEST_SIZE  = 24 * 30 * 1
GAP        = MAX_HOLD

oos_probs = []
oos_labels = []
oos_index = []

start = 0

while True:

    train_end = start + TRAIN_SIZE
    test_start = train_end + GAP
    test_end = test_start + TEST_SIZE

    if test_end >= len(X):
        break

    X_train = X.iloc[start:train_end]
    y_train = y.iloc[start:train_end]

    X_test = X.iloc[test_start:test_end]
    y_test = y.iloc[test_start:test_end]

    # =========================
    # MODEL
    # =========================
    model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=300,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=3,
        gamma=1,
        tree_method="hist",
        random_state=42,
        eval_metric="mlogloss"
    )

    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)

    oos_probs.append(probs)
    oos_labels.extend(y_test.values)
    oos_index.extend(X_test.index)

    start += TEST_SIZE


# =========================
# CONCAT OOS RESULTS
# =========================
oos_probs = np.vstack(oos_probs)

oos_df = pd.DataFrame(
    oos_probs,
    index=oos_index,
    columns=["prob_short", "prob_neutral", "prob_long"]
)

oos_labels = pd.Series(oos_labels, index=oos_index)


# =========================
# PRINT SUMMARY
# =========================
print("\nOOS Label Distribution:")
print(oos_labels.value_counts(normalize=True))

print("\nOOS Probability Summary:")
print(oos_df.describe())

Label distribution:
label
2    0.471110
0    0.471047
1    0.057842
Name: proportion, dtype: float64

OOS Label Distribution:
2    0.475463
0    0.415278
1    0.109259
Name: proportion, dtype: float64

OOS Probability Summary:
        prob_short  prob_neutral    prob_long
count  6480.000000   6480.000000  6480.000000
mean      0.466580      0.054281     0.479140
std       0.122437      0.060667     0.122423
min       0.049646      0.001329     0.084011
25%       0.381658      0.017074     0.392328
50%       0.465587      0.032768     0.478534
75%       0.554054      0.070114     0.561405
max       0.909723      0.504733     0.948873


In [11]:
threshold = 0.70

long_signals = oos_df["prob_long"] > threshold

print("Jumlah long signal:", long_signals.sum())

winrate_long = (oos_labels[long_signals] == 2).mean()
print("Winrate long:", winrate_long)

Jumlah long signal: 242
Winrate long: 0.7272727272727273


In [17]:
import numpy as np

def sharpe_ratio_evalpy(returns):
    returns = np.array(returns)
    returns = returns[~np.isnan(returns)]
    if len(returns) == 0:
        return np.nan
    return np.mean(returns) / (np.std(returns) + 1e-9)

def sharpe_ratio_annualized_hourly(returns):
    returns = np.array(returns)
    returns = returns[~np.isnan(returns)]
    if len(returns) == 0:
        return np.nan
    return np.mean(returns) / (np.std(returns) + 1e-9) * np.sqrt(365)

# Asumsi: reward 1 jika benar, -1 jika salah, 0 jika tidak ada sinyal
long_returns = np.where(long_signals, (oos_labels == 2).astype(float)*2-1, np.nan)
sharpe_long_evalpy = sharpe_ratio_evalpy(long_returns)
sharpe_long_annual = sharpe_ratio_annualized_hourly(long_returns)
print("Sharpe ratio long (evaluation.py style, per jam):", sharpe_long_evalpy)
print("Sharpe ratio long (annualized, hourly):", sharpe_long_annual)

Sharpe ratio long (evaluation.py style, per jam): 0.5103103625069121
Sharpe ratio long (annualized, hourly): 9.749465786385766


In [10]:
short_signals = oos_df["prob_short"] > threshold

print("Jumlah short signal:", short_signals.sum())

winrate_short = (oos_labels[short_signals] == 0).mean()
print("Winrate short:", winrate_short)

Jumlah short signal: 151
Winrate short: 0.3509933774834437


In [118]:
final_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=3,
    gamma=1,
    tree_method="hist",
    random_state=42,
    eval_metric="mlogloss"
)

final_model.fit(X, y)

In [119]:
import joblib

model_key = "roms/model/btc_future_1.pkl"

model_data = {
    "model": final_model,
    "features": features,
    "label_map": {-1: 0, 0: 1, 1: 2}
}

file_name = qb.object_store.get_file_path(model_key)
joblib.dump(model_data, file_name)